# Forecasting Ordini E-Commerce

## Obiettivo
Costruire un sistema di **previsione del volume giornaliero di ordini** per un e-commerce, confrontando due approcci complementari:

| Approccio | Modelli | Tipo | Punto di forza |
|-----------|---------|------|----------------|
| **Tree-based ML** | XGBoost, LightGBM | Supervisioned regression | Cattura relazioni non-lineari tra feature |
| **Time Series Decomposition** | Prophet (Meta) | Additive model | Decompone trend, stagionalita, festivita |

## Dataset
**Online Retail Dataset** (UCI Machine Learning Repository): transazioni di un e-commerce UK tra Dic 2010 e Dic 2011.
- ~541K righe di transazioni
- 8 colonne: InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country

## Pipeline
```
Dati Grezzi -> Pulizia -> Aggregazione Giornaliera -> Feature Engineering -> Split Temporale -> Modelli -> Valutazione
```

---
## 0. Setup e Importazioni

Importiamo tutte le librerie necessarie:
- **pandas/numpy**: manipolazione dati
- **matplotlib/seaborn/plotly**: visualizzazioni (statiche e interattive)
- **scikit-learn**: metriche di valutazione e utilities
- **xgboost/lightgbm**: modelli gradient boosting
- **prophet**: modello di forecasting time series di Meta

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
import warnings
import os
import json

warnings.filterwarnings('ignore')

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import lightgbm as lgb
from prophet import Prophet
import joblib

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print('Tutte le librerie caricate correttamente.')

---
## 1. Configurazione

Definiamo i parametri principali della pipeline:
- **Path del dataset** (file Excel nella cartella `online_retail/`)
- **Date di split** per dividere i dati in Train, Validation e Test
- **Orizzonte di forecast** per Prophet (30 giorni)

In [ ]:
FILE_PATH = os.path.join('online_retail', 'Online Retail.xlsx')

TRAIN_END_DATE = '2011-10-31'
VAL_END_DATE = '2011-11-15'
FORECAST_DAYS = 30

print(f'Dataset: {FILE_PATH}')
print(f'Esiste: {os.path.exists(FILE_PATH)}')
print(f'Train fino a: {TRAIN_END_DATE}')
print(f'Validation fino a: {VAL_END_DATE}')
print(f'Test: tutto dopo {VAL_END_DATE}')

---
## 2. Caricamento Dati

Carichiamo il file Excel e convertiamo la colonna `InvoiceDate` in formato datetime per poterla manipolare.

In [ ]:
df = pd.read_excel(FILE_PATH, engine='openpyxl')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')

print(f'Dati caricati: {df.shape[0]:,} righe x {df.shape[1]} colonne')
print(f'Periodo: {df["InvoiceDate"].min()} -> {df["InvoiceDate"].max()}')
print(f'\nColonne: {list(df.columns)}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

### Valori mancanti

Verifichiamo quanti valori mancanti ci sono per ogni colonna. Questo ci aiuta a capire la qualita dei dati prima della pulizia.

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Mancanti': missing, '%': missing_pct}).query('Mancanti > 0')
print('Valori mancanti per colonna:')
missing_df

---
## 3. Pulizia Dati

Applichiamo una serie di filtri per garantire la qualita dei dati:

1. **Rimozione date NaN**: righe senza data non sono utilizzabili
2. **Rimozione resi**: le fatture che iniziano con `C` sono cancellazioni/resi e distorcono il conteggio ordini
3. **Filtro Quantity > 0**: quantita negative o zero indicano errori o aggiustamenti
4. **Filtro UnitPrice > 0**: prezzi zero o negativi sono anomalie (omaggi, errori)

Infine calcoliamo il **Revenue** (Quantity x UnitPrice) per ogni riga.

In [ ]:
print(f'Righe iniziali: {len(df):,}')

df = df.dropna(subset=['InvoiceDate'])
print(f'Dopo rimozione date NaN: {len(df):,}')

df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
print(f'Dopo rimozione resi: {len(df):,}')

df = df[df['Quantity'] > 0]
print(f'Dopo filtro Quantity > 0: {len(df):,}')

df = df[df['UnitPrice'] > 0]
print(f'Dopo filtro UnitPrice > 0: {len(df):,}')

df['Revenue'] = df['Quantity'] * df['UnitPrice']
df['Date'] = df['InvoiceDate'].dt.date

print(f'\nRecord rimossi totali: {541909 - len(df):,} ({(541909 - len(df))/541909*100:.1f}%)')

---
## 4. Analisi Esplorativa (EDA)

Prima di costruire i modelli, esploriamo i dati per capire pattern e comportamenti.

### 4.1 Serie Temporali Principali
Visualizziamo le 4 metriche chiave nel tempo: ordini, revenue, quantita e clienti unici.

In [ ]:
fig = make_subplots(rows=2, cols=2,
    subplot_titles=('Ordini Giornalieri', 'Revenue Giornaliero (GBP)', 'Quantita Giornaliera', 'Clienti Unici Giornalieri'),
    vertical_spacing=0.12, horizontal_spacing=0.08)

daily_eda = df.groupby('Date').agg(
    orders=('InvoiceNo', 'nunique'),
    revenue=('Revenue', 'sum'),
    quantity=('Quantity', 'sum'),
    customers=('CustomerID', 'nunique')
).reset_index()
daily_eda['Date'] = pd.to_datetime(daily_eda['Date'])

fig.add_trace(go.Scatter(x=daily_eda['Date'], y=daily_eda['orders'], mode='lines', name='Ordini', line=dict(color='#636EFA')), row=1, col=1)
fig.add_trace(go.Scatter(x=daily_eda['Date'], y=daily_eda['revenue'], mode='lines', name='Revenue', line=dict(color='#00CC96')), row=1, col=2)
fig.add_trace(go.Scatter(x=daily_eda['Date'], y=daily_eda['quantity'], mode='lines', name='Quantita', line=dict(color='#EF553B')), row=2, col=1)
fig.add_trace(go.Scatter(x=daily_eda['Date'], y=daily_eda['customers'], mode='lines', name='Clienti', line=dict(color='#AB63FA')), row=2, col=2)

fig.update_layout(height=600, showlegend=False, title_text='Panoramica Serie Temporali')
fig.show()

### 4.2 Distribuzione e Pattern Settimanale

A sinistra la distribuzione degli ordini giornalieri (per verificare normalita e outlier), a destra la media per giorno della settimana (per individuare pattern ciclici settimanali).

In [ ]:
daily_orders_eda = df.groupby(pd.to_datetime(df['Date']))['InvoiceNo'].nunique()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

daily_orders_eda.hist(bins=30, ax=axes[0], edgecolor='black', alpha=0.7)
axes[0].set_title('Distribuzione Ordini Giornalieri')
axes[0].set_xlabel('N. Ordini')
axes[0].axvline(daily_orders_eda.mean(), color='red', linestyle='--', label=f'Media: {daily_orders_eda.mean():.1f}')
axes[0].axvline(daily_orders_eda.median(), color='orange', linestyle='--', label=f'Mediana: {daily_orders_eda.median():.1f}')
axes[0].legend()

day_labels = {0: 'Lun', 1: 'Mar', 2: 'Mer', 3: 'Gio', 4: 'Ven', 5: 'Sab', 6: 'Dom'}
avg_by_day = daily_orders_eda.groupby(daily_orders_eda.index.dayofweek).mean()
avg_by_day.index = [day_labels[d] for d in avg_by_day.index]
avg_by_day.plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='black')
axes[1].set_title('Media Ordini per Giorno della Settimana')
axes[1].set_ylabel('Media Ordini')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

### 4.3 Pattern Mensile e Trend

Analizziamo la stagionalita mensile e il trend complessivo con una media mobile a 7 giorni per smussare il rumore.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

monthly_orders = daily_orders_eda.groupby(daily_orders_eda.index.month).mean()
mesi_labels = ['Gen', 'Feb', 'Mar', 'Apr', 'Mag', 'Giu', 'Lug', 'Ago', 'Set', 'Ott', 'Nov', 'Dic']
monthly_orders.index = [mesi_labels[m-1] for m in monthly_orders.index]
colors = ['#00CC96' if v > daily_orders_eda.mean() else '#EF553B' for v in monthly_orders.values]
monthly_orders.plot(kind='bar', ax=axes[0], color=colors, edgecolor='black')
axes[0].axhline(daily_orders_eda.mean(), color='gray', linestyle='--', alpha=0.7, label='Media globale')
axes[0].set_title('Media Ordini per Mese')
axes[0].set_ylabel('Media Ordini')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend()

axes[1].plot(daily_orders_eda.index, daily_orders_eda.values, alpha=0.3, label='Ordini giornalieri')
axes[1].plot(daily_orders_eda.index, daily_orders_eda.rolling(7).mean(), color='red', linewidth=2, label='Media mobile 7gg')
axes[1].set_title('Trend con Media Mobile a 7 Giorni')
axes[1].set_ylabel('N. Ordini')
axes[1].legend()

plt.tight_layout()
plt.show()

### 4.4 Matrice di Correlazione

Verifichiamo le correlazioni tra le metriche aggregate giornaliere. Correlazioni forti tra feature aiutano i modelli, ma attenzione alla multicollinearita.

In [ ]:
corr_cols = ['orders', 'revenue', 'quantity', 'customers']
corr_matrix = daily_eda[corr_cols].corr()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            square=True, linewidths=1, ax=ax,
            xticklabels=['Ordini', 'Revenue', 'Quantita', 'Clienti'],
            yticklabels=['Ordini', 'Revenue', 'Quantita', 'Clienti'])
ax.set_title('Correlazione tra Metriche Giornaliere', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.5 Distribuzione Revenue e Top Paesi

Uno sguardo alla distribuzione geografica e ai top prodotti per capire meglio il business.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top_countries = df.groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(10)
top_countries.plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Top 10 Paesi per Revenue')
axes[0].set_xlabel('Revenue Totale (GBP)')
axes[0].invert_yaxis()

daily_revenue = df.groupby('Date')['Revenue'].sum()
axes[1].boxplot([daily_revenue.values], vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue'))
axes[1].set_title('Distribuzione Revenue Giornaliero')
axes[1].set_ylabel('Revenue (GBP)')
axes[1].set_xticklabels(['Revenue/giorno'])

plt.tight_layout()
plt.show()

---
## 5. Aggregazione Giornaliera

Aggreghiamo le transazioni a livello giornaliero. Per ogni giorno calcoliamo:

| Metrica | Aggregazione | Descrizione |
|---------|-------------|-------------|
| **Orders** | `nunique(InvoiceNo)` | Numero di ordini unici |
| **TotalQuantity** | `sum(Quantity)` | Pezzi totali venduti |
| **Revenue** | `sum(Revenue)` | Fatturato totale |
| **UniqueCustomers** | `nunique(CustomerID)` | Clienti distinti |
| **UniqueProducts** | `nunique(StockCode)` | Prodotti distinti venduti |

La variabile **target** che vogliamo predire e `Orders` (numero di ordini giornalieri).

In [ ]:
daily = df.groupby('Date').agg({
    'InvoiceNo': 'nunique',
    'Quantity': 'sum',
    'Revenue': 'sum',
    'CustomerID': 'nunique',
    'StockCode': 'nunique'
}).reset_index()

daily.columns = ['Date', 'Orders', 'TotalQuantity', 'Revenue', 'UniqueCustomers', 'UniqueProducts']
daily['Date'] = pd.to_datetime(daily['Date'])
daily = daily.sort_values('Date').reset_index(drop=True)

print(f'Giorni totali: {len(daily)}')
print(f'Media ordini/giorno: {daily["Orders"].mean():.1f}')
print(f'Deviazione standard: {daily["Orders"].std():.1f}')
print(f'Min ordini: {daily["Orders"].min()}, Max ordini: {daily["Orders"].max()}')
daily.head(10)

---
## 6. Feature Engineering

Creiamo 30 feature a partire dalla data e dai valori passati. Questo e il cuore della pipeline ML.

### Categorie di Feature:

**Feature Temporali (15)**: estraggono informazioni dal calendario
- Componenti base: Year, Month, Day, DayOfWeek, DayOfYear, WeekOfYear, Quarter
- **Encoding ciclico** (sin/cos): permette al modello di capire che Dicembre e Gennaio sono "vicini" (non lontani come 12 vs 1). Stessa logica per i giorni della settimana.
- **Flag binari**: IsWeekend, IsMonthStart (primi 5gg), IsMonthEnd (ultimi 6gg)

**Lag Features (6)**: il valore degli ordini nei giorni precedenti
- Lag 1, 2, 3: catturano l'inerzia a breve termine
- Lag 7: cattura il pattern settimanale
- Lag 14, 28: catturano pattern bisettimanali e mensili

**Rolling Features (6)**: statistiche calcolate su finestre mobili
- Media e deviazione standard su 7, 14, 28 giorni
- Lo `shift(1)` evita **data leakage**: usiamo solo dati disponibili al momento della previsione

In [ ]:
# Feature temporali
daily['Year'] = daily['Date'].dt.year
daily['Month'] = daily['Date'].dt.month
daily['Day'] = daily['Date'].dt.day
daily['DayOfWeek'] = daily['Date'].dt.dayofweek
daily['DayOfYear'] = daily['Date'].dt.dayofyear
daily['WeekOfYear'] = daily['Date'].dt.isocalendar().week
daily['Quarter'] = daily['Date'].dt.quarter

# Encoding ciclico
daily['DayOfWeek_sin'] = np.sin(2 * np.pi * daily['DayOfWeek'] / 7)
daily['DayOfWeek_cos'] = np.cos(2 * np.pi * daily['DayOfWeek'] / 7)
daily['Month_sin'] = np.sin(2 * np.pi * daily['Month'] / 12)
daily['Month_cos'] = np.cos(2 * np.pi * daily['Month'] / 12)

# Flag
daily['IsWeekend'] = (daily['DayOfWeek'] >= 5).astype(int)
daily['IsMonthStart'] = (daily['Day'] <= 5).astype(int)
daily['IsMonthEnd'] = (daily['Day'] >= 25).astype(int)

# Lag features
lags = [1, 2, 3, 7, 14, 28]
for lag in lags:
    daily[f'Orders_lag_{lag}'] = daily['Orders'].shift(lag)

# Rolling features (con shift per evitare data leakage)
windows = [7, 14, 28]
for window in windows:
    daily[f'Orders_rolling_mean_{window}'] = daily['Orders'].shift(1).rolling(window=window).mean()
    daily[f'Orders_rolling_std_{window}'] = daily['Orders'].shift(1).rolling(window=window).std()

daily_clean = daily.dropna().reset_index(drop=True)

print(f'Feature temporali: 15')
print(f'Lag features: {len(lags)}')
print(f'Rolling features: {len(windows) * 2}')
print(f'Feature dal dataset: 4 (TotalQuantity, Revenue, UniqueCustomers, UniqueProducts)')
print(f'\nRighe perse per lag/rolling NaN: {len(daily) - len(daily_clean)}')
print(f'Righe finali: {len(daily_clean)}')
print(f'Feature totali: {len(daily_clean.columns) - 2}')

### Visualizzazione Encoding Ciclico

L'encoding ciclico trasforma valori lineari (es. mese 1-12) in coordinate su un cerchio usando sin e cos.  
Questo permette al modello di capire che **Dicembre (12) e Gennaio (1) sono adiacenti**, cosa impossibile con un encoding lineare.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

theta_dow = np.linspace(0, 2*np.pi, 7, endpoint=False)
days = ['Lun', 'Mar', 'Mer', 'Gio', 'Ven', 'Sab', 'Dom']
axes[0].scatter(np.sin(theta_dow), np.cos(theta_dow), s=100, zorder=5)
for i, d in enumerate(days):
    axes[0].annotate(d, (np.sin(theta_dow[i]), np.cos(theta_dow[i])), fontsize=11, ha='center', va='bottom', xytext=(0, 8), textcoords='offset points')
circle = plt.Circle((0, 0), 1, fill=False, linestyle='--', alpha=0.3)
axes[0].add_patch(circle)
axes[0].set_xlim(-1.4, 1.4)
axes[0].set_ylim(-1.4, 1.4)
axes[0].set_aspect('equal')
axes[0].set_title('Encoding Ciclico - Giorni della Settimana')
axes[0].set_xlabel('sin(DayOfWeek)')
axes[0].set_ylabel('cos(DayOfWeek)')
axes[0].grid(True, alpha=0.3)

theta_m = np.linspace(0, 2*np.pi, 12, endpoint=False)
mesi_short = ['Gen', 'Feb', 'Mar', 'Apr', 'Mag', 'Giu', 'Lug', 'Ago', 'Set', 'Ott', 'Nov', 'Dic']
axes[1].scatter(np.sin(theta_m), np.cos(theta_m), s=100, zorder=5, color='orange')
for i, m in enumerate(mesi_short):
    axes[1].annotate(m, (np.sin(theta_m[i]), np.cos(theta_m[i])), fontsize=10, ha='center', va='bottom', xytext=(0, 8), textcoords='offset points')
circle2 = plt.Circle((0, 0), 1, fill=False, linestyle='--', alpha=0.3)
axes[1].add_patch(circle2)
axes[1].set_xlim(-1.4, 1.4)
axes[1].set_ylim(-1.4, 1.4)
axes[1].set_aspect('equal')
axes[1].set_title('Encoding Ciclico - Mesi')
axes[1].set_xlabel('sin(Month)')
axes[1].set_ylabel('cos(Month)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 7. Split Train / Validation / Test

Per le serie temporali **non possiamo fare random split** (causerebbe data leakage). Usiamo uno **split cronologico**:

```
|<---------- TRAIN ---------->|<-- VAL -->|<----- TEST ----->|
  Gen 2011          Ott 2011   Nov 1-15      Nov 16 - Dic 9
       243 giorni               13 giorni       21 giorni
```

- **Train**: il modello impara i pattern
- **Validation**: si seleziona il miglior modello e si tuning gli iperparametri
- **Test**: valutazione finale su dati mai visti (simula il futuro)

In [ ]:
train_end = pd.to_datetime(TRAIN_END_DATE)
val_end = pd.to_datetime(VAL_END_DATE)

train = daily_clean[daily_clean['Date'] <= train_end].copy()
val = daily_clean[(daily_clean['Date'] > train_end) & (daily_clean['Date'] <= val_end)].copy()
test = daily_clean[daily_clean['Date'] > val_end].copy()

exclude_cols = ['Date', 'Orders']
feature_cols = [col for col in daily_clean.columns if col not in exclude_cols]

X_train, y_train = train[feature_cols], train['Orders']
X_val, y_val = val[feature_cols], val['Orders']
X_test, y_test = test[feature_cols], test['Orders']

print(f'Train: {len(train)} giorni ({train["Date"].min().date()} -> {train["Date"].max().date()})')
print(f'Val:   {len(val)} giorni ({val["Date"].min().date()} -> {val["Date"].max().date()})')
print(f'Test:  {len(test)} giorni ({test["Date"].min().date()} -> {test["Date"].max().date()})')
print(f'Features: {len(feature_cols)}')

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=train['Date'], y=y_train, name='Train', mode='lines', fill='tozeroy', opacity=0.7))
fig.add_trace(go.Scatter(x=val['Date'], y=y_val, name='Validation', mode='lines', fill='tozeroy', opacity=0.7))
fig.add_trace(go.Scatter(x=test['Date'], y=y_test, name='Test', mode='lines', fill='tozeroy', opacity=0.7))

fig.add_shape(type='line', x0=str(train_end.date()), x1=str(train_end.date()), y0=0, y1=1, yref='paper', line=dict(dash='dash', color='gray'))
fig.add_shape(type='line', x0=str(val_end.date()), x1=str(val_end.date()), y0=0, y1=1, yref='paper', line=dict(dash='dash', color='gray'))
fig.add_annotation(x=str(train_end.date()), y=1, yref='paper', text='Fine Train', showarrow=False, yanchor='bottom')
fig.add_annotation(x=str(val_end.date()), y=1, yref='paper', text='Fine Val', showarrow=False, yanchor='bottom')

fig.update_layout(title='Split Temporale dei Dati', xaxis_title='Data', yaxis_title='Ordini', height=400)
fig.show()

---
## 8. Baseline Models

Prima di usare modelli complessi, definiamo delle **baseline semplici**. Servono come riferimento: se un modello ML non batte le baseline, non vale la complessita aggiunta.

| Baseline | Logica | Quando funziona bene |
|----------|--------|---------------------|
| **Mean** | Predice sempre la media storica del training | Serie stazionarie senza trend |
| **Last Value** | Predice sempre l'ultimo valore osservato | Serie con forte inerzia |
| **MA(7)** | Predice la media degli ultimi 7 giorni | Serie con pattern settimanale |

In [ ]:
def evaluate(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return {'Model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': f'{mape:.2f}%', 'R2': f'{r2:.4f}'}

baselines = []

preds_mean = np.full(len(val), train['Orders'].mean())
baselines.append(evaluate(y_val, preds_mean, 'Baseline Mean'))

preds_last = np.full(len(val), train['Orders'].iloc[-1])
baselines.append(evaluate(y_val, preds_last, 'Baseline Last'))

preds_ma7 = np.full(len(val), train['Orders'].tail(7).mean())
baselines.append(evaluate(y_val, preds_ma7, 'Baseline MA(7)'))

pd.DataFrame(baselines)

### Interpretazione Metriche

| Metrica | Significato | Interpretazione |
|---------|------------|----------------|
| **MAE** | Errore medio assoluto | "In media sbaglio di X ordini" |
| **RMSE** | Root Mean Square Error | Come MAE ma penalizza di piu gli errori grandi |
| **MAPE** | % errore medio | "In media sbaglio del X%" |
| **R2** | Coefficiente di determinazione | 1.0 = perfetto, 0.0 = come la media, <0 = peggio della media |

---
## 9. XGBoost

**XGBoost** (eXtreme Gradient Boosting) costruisce un ensemble di alberi decisionali in sequenza, dove ogni nuovo albero corregge gli errori del precedente.

**Iperparametri scelti:**
- `max_depth=6`: profondita massima degli alberi (bilancia complessita vs overfitting)
- `learning_rate=0.1`: quanto "impara" ogni albero (piu basso = piu conservativo)
- `n_estimators=100`: numero di alberi
- `subsample=0.8`: usa l'80% dei dati per ogni albero (riduce overfitting)
- `colsample_bytree=0.8`: usa l'80% delle feature per ogni albero

In [ ]:
xgb_params = {
    'objective': 'reg:squarederror',
    'max_depth': 6,
    'learning_rate': 0.1,
    'n_estimators': 100,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42
}

model_xgb = xgb.XGBRegressor(**xgb_params)
model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

preds_xgb = model_xgb.predict(X_val)
res_xgb = evaluate(y_val, preds_xgb, 'XGBoost')

print('XGBoost - Validation Results:')
for k, v in res_xgb.items():
    if k != 'Model':
        print(f'  {k}: {v}')

---
## 10. LightGBM

**LightGBM** e un'alternativa a XGBoost sviluppata da Microsoft. Usa un algoritmo di crescita degli alberi "leaf-wise" (anziche "level-wise") che puo essere piu veloce e preciso su dataset grandi.

Usiamo gli stessi iperparametri per un confronto equo, con l'aggiunta di **early stopping** (ferma il training quando le performance sul validation smettono di migliorare).

In [ ]:
lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'max_depth': 6,
    'learning_rate': 0.1,
    'n_estimators': 100,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'verbose': -1
}

model_lgb = lgb.LGBMRegressor(**lgb_params)
model_lgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=10, verbose=False)]
)

preds_lgb = model_lgb.predict(X_val)
res_lgb = evaluate(y_val, preds_lgb, 'LightGBM')

print('LightGBM - Validation Results:')
for k, v in res_lgb.items():
    if k != 'Model':
        print(f'  {k}: {v}')

---
## 11. Confronto Modelli

Confrontiamo tutti i modelli sulla base del RMSE sul validation set. Il modello con RMSE piu basso viene selezionato come "best model".

In [ ]:
all_results = baselines + [res_xgb, res_lgb]
results_df = pd.DataFrame(all_results)

rmse_xgb = float(res_xgb['RMSE'])
rmse_lgb = float(res_lgb['RMSE'])
best_model_name = 'XGBoost' if rmse_xgb < rmse_lgb else 'LightGBM'
best_model = model_xgb if rmse_xgb < rmse_lgb else model_lgb
best_preds = preds_xgb if rmse_xgb < rmse_lgb else preds_lgb

print(f'BEST MODEL: {best_model_name} (RMSE: {min(rmse_xgb, rmse_lgb):.2f})')
results_df

In [ ]:
best_rmse_val = min(rmse_xgb, rmse_lgb)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=results_df['Model'],
    y=results_df['RMSE'],
    marker_color=['#ef553b' if r > best_rmse_val else '#00cc96' for r in results_df['RMSE']],
    text=[f'{r:.1f}' for r in results_df['RMSE']],
    textposition='outside'
))
fig.update_layout(title='Confronto RMSE tra Modelli (piu basso = migliore)', yaxis_title='RMSE', height=420)
fig.show()

---
## 12. Visualizzazioni Dettagliate

### 12.1 Predizioni vs Reali (Validation)

Confronto diretto tra i valori reali e le predizioni del miglior modello sul validation set.

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=val['Date'], y=y_val.values, name='Valori Reali', mode='lines+markers', marker=dict(size=8)))
fig.add_trace(go.Scatter(x=val['Date'], y=best_preds, name=f'Predizioni {best_model_name}', mode='lines+markers', line=dict(dash='dash'), marker=dict(size=8, symbol='x')))

for i in range(len(val)):
    fig.add_trace(go.Scatter(
        x=[val['Date'].iloc[i], val['Date'].iloc[i]],
        y=[y_val.values[i], best_preds[i]],
        mode='lines', line=dict(color='rgba(255,0,0,0.3)', width=1),
        showlegend=False
    ))

fig.update_layout(title=f'Validation Set - {best_model_name}: Predizioni vs Reali (le linee rosse mostrano l\'errore)',
                  xaxis_title='Data', yaxis_title='Ordini', height=450)
fig.show()

### 12.2 Feature Importance

Quali feature contribuiscono di piu alle predizioni? Questo aiuta a capire cosa "guida" il modello e puo suggerire nuove feature da creare o feature inutili da rimuovere.

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    indices = np.argsort(importances)[::-1][:15]
    top_features = [feature_cols[i] for i in indices]
    top_importances = importances[indices]

    fig = go.Figure(go.Bar(
        x=top_importances[::-1],
        y=top_features[::-1],
        orientation='h',
        marker_color='steelblue',
        text=[f'{v:.1f}' for v in top_importances[::-1]],
        textposition='outside'
    ))
    fig.update_layout(title=f'Top 15 Feature Importance ({best_model_name})', xaxis_title='Importanza', height=500)
    fig.show()

### 12.3 Confronto XGBoost vs LightGBM (Validation)

Vediamo come si comportano i due modelli sugli stessi dati di validation.

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=val['Date'], y=y_val.values, name='Reali', mode='lines+markers', line=dict(color='black', width=2)))
fig.add_trace(go.Scatter(x=val['Date'], y=preds_xgb, name='XGBoost', mode='lines+markers', line=dict(dash='dash')))
fig.add_trace(go.Scatter(x=val['Date'], y=preds_lgb, name='LightGBM', mode='lines+markers', line=dict(dash='dot')))
fig.update_layout(title='XGBoost vs LightGBM vs Reali (Validation)', xaxis_title='Data', yaxis_title='Ordini', height=420)
fig.show()

---
## 13. Valutazione su Test Set

La valutazione finale viene fatta sul **test set**, che il modello non ha mai visto ne durante il training ne durante la selezione. Questo simula il comportamento del modello nel mondo reale.

In [ ]:
preds_test = best_model.predict(X_test)
res_test = evaluate(y_test, preds_test, f'{best_model_name} (Test)')

print(f'{best_model_name} - Test Set Results:')
for k, v in res_test.items():
    if k != 'Model':
        print(f'  {k}: {v}')

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=train['Date'], y=y_train.values, name='Train', mode='lines', opacity=0.4))
fig.add_trace(go.Scatter(x=val['Date'], y=y_val.values, name='Validation (reali)', mode='lines'))
fig.add_trace(go.Scatter(x=val['Date'], y=best_preds, name=f'Validation (pred)', mode='lines', line=dict(dash='dash')))
fig.add_trace(go.Scatter(x=test['Date'], y=y_test.values, name='Test (reali)', mode='lines'))
fig.add_trace(go.Scatter(x=test['Date'], y=preds_test, name=f'Test (pred)', mode='lines', line=dict(dash='dash')))

fig.add_shape(type='line', x0=str(train_end.date()), x1=str(train_end.date()), y0=0, y1=1, yref='paper', line=dict(dash='dash', color='gray', width=1), opacity=0.5)
fig.add_shape(type='line', x0=str(val_end.date()), x1=str(val_end.date()), y0=0, y1=1, yref='paper', line=dict(dash='dash', color='gray', width=1), opacity=0.5)

fig.update_layout(title=f'Vista Completa: {best_model_name} su Train + Validation + Test', xaxis_title='Data', yaxis_title='Ordini', height=500)
fig.show()

---
## 14. Analisi Residui

I **residui** (errore = reale - predetto) ci dicono dove e come il modello sbaglia:

- **Residui vs Predizioni**: se c'e un pattern, il modello ha bias sistematico
- **Distribuzione residui**: idealmente dovrebbero essere centrati su zero e simmetrici (errori casuali)
- **Residui nel tempo**: se c'e un trend, il modello sta deteriorando

In [ ]:
residuals_val = y_val.values - best_preds
residuals_test = y_test.values - preds_test

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(best_preds, residuals_val, alpha=0.7, label='Validation', s=60)
axes[0].scatter(preds_test, residuals_test, alpha=0.7, label='Test', s=60)
axes[0].axhline(y=0, color='red', linestyle='--')
axes[0].set_xlabel('Predizioni')
axes[0].set_ylabel('Residui (reale - pred)')
axes[0].set_title('Residui vs Predizioni')
axes[0].legend()

axes[1].hist(residuals_test, bins=15, edgecolor='black', alpha=0.7, color='steelblue')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_title(f'Distribuzione Residui (Test)\nMedia: {residuals_test.mean():.1f}, Std: {residuals_test.std():.1f}')
axes[1].set_xlabel('Residuo')

axes[2].plot(test['Date'], residuals_test, marker='o', linewidth=1)
axes[2].axhline(y=0, color='red', linestyle='--')
axes[2].fill_between(test['Date'], residuals_test, 0, alpha=0.2)
axes[2].set_title('Residui nel Tempo (Test)')
axes[2].set_ylabel('Residuo')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

---
## 15. Prophet - Time Series Forecasting

**Prophet** (sviluppato da Meta) adotta un approccio completamente diverso dai modelli ML:

$$y(t) = g(t) + s(t) + h(t) + \epsilon_t$$

| Componente | Significato |
|------------|------------|
| $g(t)$ | **Trend**: la crescita/decrescita generale del business |
| $s(t)$ | **Stagionalita**: pattern che si ripetono (settimanale, annuale) |
| $h(t)$ | **Festivita**: effetti di giorni speciali |
| $\epsilon_t$ | **Rumore**: variazione casuale non prevedibile |

A differenza di XGBoost/LightGBM, Prophet lavora direttamente sulla serie temporale senza bisogno di feature engineering manuale.

**Nota**: qui previediamo la **quantita totale** (non il numero ordini) per mostrare un uso diverso del modello.

In [ ]:
df_prophet = df.copy()
daily_sales = df_prophet.groupby(df_prophet['InvoiceDate'].dt.date)['Quantity'].sum().reset_index()
daily_sales.columns = ['ds', 'y']
daily_sales['ds'] = pd.to_datetime(daily_sales['ds'])

print(f'Giorni di dati per Prophet: {len(daily_sales)}')
print(f'Periodo: {daily_sales["ds"].min()} -> {daily_sales["ds"].max()}')
print(f'Media quantita/giorno: {daily_sales["y"].mean():,.0f}')

In [ ]:
model_prophet = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05
)
model_prophet.fit(daily_sales)
print('Prophet addestrato.')

### 15.1 Forecast e Componenti

Il grafico del forecast mostra:
- **Punti neri**: dati reali osservati
- **Linea blu**: previsione del modello
- **Area azzurra**: intervallo di confidenza (incertezza della previsione)

In [ ]:
future = model_prophet.make_future_dataframe(periods=FORECAST_DAYS)
forecast = model_prophet.predict(future)

fig1 = model_prophet.plot(forecast)
plt.title(f'Prophet Forecast - Prossimi {FORECAST_DAYS} Giorni', fontsize=14, pad=20)
plt.ylabel('Quantita Ordinata')
plt.xlabel('Data')
plt.tight_layout()
plt.show()

### 15.2 Decomposizione in Componenti

Prophet scompone la serie in trend + stagionalita settimanale + stagionalita annuale. Questo e molto utile per capire i **driver** del business.

In [ ]:
fig2 = model_prophet.plot_components(forecast)
plt.tight_layout()
plt.show()

### 15.3 Pattern Settimanale

Quanto ogni giorno della settimana si discosta dalla media? Valori positivi = giorno sopra la media, negativi = sotto.

In [ ]:
giorni = ['Lunedi', 'Martedi', 'Mercoledi', 'Giovedi', 'Venerdi', 'Sabato', 'Domenica']
week_dates = pd.date_range('2023-01-02', periods=7)
week_df = pd.DataFrame({'ds': week_dates})
weekly_effect = model_prophet.predict_seasonal_components(model_prophet.setup_dataframe(week_df))['weekly'].values

fig = go.Figure(go.Bar(
    x=giorni,
    y=weekly_effect,
    marker_color=['#00cc96' if e > 0 else '#ef553b' for e in weekly_effect],
    text=[f'{e:+,.0f}' for e in weekly_effect],
    textposition='outside'
))
fig.update_layout(title='Prophet - Pattern Settimanale (effetto vs media giornaliera)', yaxis_title='Effetto (unita)', height=420)
fig.show()

print(f'Giorno MIGLIORE: {giorni[np.argmax(weekly_effect)]} ({weekly_effect.max():+,.0f} unita vs media)')
print(f'Giorno PEGGIORE: {giorni[np.argmin(weekly_effect)]} ({weekly_effect.min():+,.0f} unita vs media)')

### 15.4 Pattern Annuale

Quanto ogni mese si discosta dalla media annuale? Questo rivela la **stagionalita del business**.

In [ ]:
mesi = ['Gen', 'Feb', 'Mar', 'Apr', 'Mag', 'Giu', 'Lug', 'Ago', 'Set', 'Ott', 'Nov', 'Dic']
year_dates = pd.date_range('2023-01-01', periods=365)
year_df = pd.DataFrame({'ds': year_dates})
yearly_effect = model_prophet.predict_seasonal_components(model_prophet.setup_dataframe(year_df))['yearly'].values

monthly_effect = []
for i in range(12):
    month_start = i * 30
    month_end = min((i + 1) * 30, len(yearly_effect))
    monthly_effect.append(yearly_effect[month_start:month_end].mean())

fig = go.Figure(go.Bar(
    x=mesi,
    y=monthly_effect,
    marker_color=['#00cc96' if e > 0 else '#ef553b' for e in monthly_effect],
    text=[f'{e:+,.0f}' for e in monthly_effect],
    textposition='outside'
))
fig.update_layout(title='Prophet - Pattern Annuale (effetto vs media mensile)', yaxis_title='Effetto (unita)', height=420)
fig.show()

print(f'Mese MIGLIORE: {mesi[np.argmax(monthly_effect)]} ({max(monthly_effect):+,.0f} unita vs media)')
print(f'Mese PEGGIORE: {mesi[np.argmin(monthly_effect)]} ({min(monthly_effect):+,.0f} unita vs media)')

### 15.5 Trend e Punti di Svolta

Analizziamo il trend generale del business e i **changepoint** (momenti in cui il tasso di crescita cambia).

In [ ]:
trend_start = forecast['trend'].iloc[0]
trend_end = forecast['trend'].iloc[-1]
trend_change = trend_end - trend_start
trend_pct = (trend_change / trend_start) * 100

fig = go.Figure()
fig.add_trace(go.Scatter(x=forecast['ds'], y=forecast['trend'], name='Trend', line=dict(color='#636EFA', width=3)))

if hasattr(model_prophet, 'changepoints'):
    for cp in model_prophet.changepoints:
        fig.add_vline(x=cp, line_dash='dot', line_color='red', opacity=0.3)

fig.update_layout(
    title=f'Prophet - Trend del Business ({"Crescita" if trend_change > 0 else "Decrescita"}: {trend_pct:+.1f}%)',
    xaxis_title='Data', yaxis_title='Trend (unita)', height=400,
    annotations=[dict(text='Linee rosse = changepoints', xref='paper', yref='paper', x=0.02, y=0.98, showarrow=False, font=dict(size=11, color='red'))]
)
fig.show()

### 15.6 Previsioni Prossimi 7 Giorni

Le previsioni finali di Prophet con intervallo di confidenza e suggerimento per il safety stock.

In [ ]:
next_week = forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(7).copy()
next_week['ds'] = next_week['ds'].dt.strftime('%Y-%m-%d')
next_week.columns = ['Data', 'Previsione', 'Min (CI 80%)', 'Max (CI 80%)']

print('Previsioni Prophet - Prossimi 7 Giorni:')
print(next_week.to_string(index=False))

avg_fc = next_week['Previsione'].mean()
print(f'\nMedia prevista: {avg_fc:,.0f} unita/giorno')
print(f'Scenario pessimistico: {next_week["Min (CI 80%)"].mean():,.0f} unita/giorno')
print(f'Scenario ottimistico: {next_week["Max (CI 80%)"].mean():,.0f} unita/giorno')
print(f'Safety stock suggerito: {next_week["Max (CI 80%)"].mean() * 1.1:,.0f} unita/giorno (+10%)')

---
## 16. Salvataggio Modello e Artefatti

Salviamo il miglior modello ML in formato pickle (`.pkl`) per poterlo riutilizzare senza ri-allenarlo. Salviamo anche la lista delle feature necessarie per l'inferenza.

In [ ]:
model_filename = f'model_{best_model_name.lower()}.pkl'
joblib.dump(best_model, model_filename)
print(f'Modello salvato: {model_filename}')

with open('feature_columns.json', 'w') as f:
    json.dump(feature_cols, f)
print('Feature columns salvate: feature_columns.json')

---
## 17. Riepilogo Finale

### Confronto Approcci

| Aspetto | XGBoost/LightGBM | Prophet |
|---------|------------------|---------|
| **Input** | Feature engineered | Serie temporale grezza |
| **Output** | Predizione puntuale | Predizione + intervalli di confidenza |
| **Interpretabilita** | Feature importance | Decomposizione trend/stagionalita |
| **Forza** | Cattura relazioni complesse | Gestisce automaticamente stagionalita |
| **Uso ideale** | Previsione precisa a breve termine | Analisi pattern + forecast a medio termine |

In [ ]:
print('=' * 60)
print('RIEPILOGO FINALE')
print('=' * 60)

print(f'\n--- Modelli ML (target: ordini giornalieri) ---')
print(f'  Miglior Modello: {best_model_name}')
print(f'  Validation RMSE: {min(rmse_xgb, rmse_lgb):.2f}')
print(f'  Test RMSE: {float(res_test["RMSE"]):.2f}')
print(f'  Test R2: {res_test["R2"]}')
print(f'  Test MAPE: {res_test["MAPE"]}')

print(f'\n--- Prophet (target: quantita giornaliera) ---')
print(f'  Trend: {"Crescita" if trend_change > 0 else "Decrescita"} ({trend_pct:+.1f}%)')
print(f'  Forecast medio 7gg: {avg_fc:,.0f} unita/giorno')
print(f'  Giorno migliore: {giorni[np.argmax(weekly_effect)]}')
print(f'  Mese migliore: {mesi[np.argmax(monthly_effect)]}')

print(f'\n--- File generati ---')
print(f'  {model_filename}')
print(f'  feature_columns.json')
print('\nProcesso completato.')